# MySpotify — complete recommendation pipeline

Merged version of notebooks 01 (Top-250), 02 (Top-100 by genre), 03 (Collections)
and 04 (Collaborative filtering).

All tables are parsed **in place from the uncleaned raw files** (`data/raw`)
with pandas — no cleaning, no intermediate files are written or read. The heavy triplets table (~48.4 M rows) and the tidy
lyrics table (~210 k tracks) are ordinary DataFrames, so every task is plain
pandas + scikit-learn + implicit.

In [1]:
import gc
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from implicit.evaluation import train_test_split as implicit_split

from src.data.loader import MySpotifyRecommender
from src.models.collaborative_filtering import (
    build_user_item_matrix,
    evaluate_user_cf,
    fit_als,
    recommend_tracks_df,
    recommend_users_df,
)
from src.models.collections import (
    collection_baseline,
    collection_classification_compare,
    collection_word2vec,
)
from src.models.top_n import top_n_songs
from src.models.top_n_genre import top_n_per_genre


def rss_mb():
    try:
        with open("/proc/self/status") as fh:
            for line in fh:
                if line.startswith("VmRSS"):
                    return int(line.split()[1]) // 1024
    except OSError:
        pass
    return 0


rs = MySpotifyRecommender.from_files(Path.cwd().parent / "data")
print(f"peak RSS : {rss_mb()} MB (all tables loaded in memory)")

/home/samy/MySpotify/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/samy/MySpotify/.venv/lib/python3.12/site-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


Data dir    : /home/samy/MySpotify/data
Source      : uncleaned raw files (parsed in place)

  tracks      (1000000, 4)
  genres      (280831, 3)
  triplets    (48373586, 3)
  lyrics_long (16845822, 3)
peak RSS : 5146 MB (all tables loaded in memory)


---
## 1. Top-250 most-played songs

Total play count per song is aggregated from the full triplets table with a
pandas ``groupby``; the 250 most-played songs are then joined to the tracks.


In [2]:
a = top_n_songs(rs, n=250, user_id=None)
print("head(5):")
display(a[["artist", "title", "play_count"]].head(5))
print("\ntail(5):")
display(a[["artist", "title", "play_count"]].tail(5))
print(f"peak RSS : {rss_mb()} MB")

head(5):


,artist,title,play_count
0,Dwight Yoakam,You're The One,726885
1,Björk,Undo,648239
2,Kings Of Leon,Revelry,527893
3,Harmonia,Sehr kosmisch,425463
4,Barry Tuckwell/Academy of St Martin-in-the-Fie...,Horn Concerto No. 4 in E flat K495: II. Romanc...,389880



tail(5):


,artist,title,play_count
245,Triple Six Mafia,Now I'm High_ Really High,35253
246,The Red Jumpsuit Apparatus,Face Down (Album Version),35245
247,Linkin Park,New Divide (Album Version),35191
248,Selena Gomez & The Scene,Naturally,35074
249,Creedence Clearwater Revival,Have You Ever Seen The Rain,34831


peak RSS : 5310 MB


In [3]:
# the same query for a single user (read from one row of the sparse matrix)
user_id = rs.triplets["user_id"].unique()[0]
print(f"sample user : {user_id}")

a = top_n_songs(rs, n=250, user_id=user_id)
display(a[["artist", "title", "play_count"]].head(10))

sample user : b80344d063b5ccb3212f76538f3d9e43d87dca9e


,artist,title,play_count
0,Jack Johnson,Moonshine,8
1,Panic At The Disco,Behind The Sea [Live In Chicago],6
2,Bobby Freeman,Do You Wanna Dance,6
3,Héroes del Silencio,Apuesta Por El Rock 'N' Roll,5
4,Puff Daddy,I'll Be Missing You (Featuring Faith Evans & 1...,5
5,Robert Johnson,I?'m A Steady Rollin? Man,5
6,Bob Rivers,No So Silent Night (album version),5
7,Paco De Lucia,Entre Dos Aguas,2
8,Jorge Drexler,12 segundos de oscuridad,2
9,Chris Bell,Speed Of Sound,2


---
## 2. Top-100 most-played songs by genre

Genres use the `majority_genre` tag. Tracks are deduplicated by `song_id`
(keeping the first row) *before* the genre join, matching the reference output.


In [4]:
genres = rs.genres["majority_genre"].unique()
print(f"Available genres ({len(genres)}): {sorted(genres)}")

Available genres (15): ['Blues', 'Country', 'Electronic', 'Folk', 'Jazz', 'Latin', 'Metal', 'New Age', 'Pop', 'Punk', 'Rap', 'Reggae', 'RnB', 'Rock', 'World']


In [5]:
top100_rock = top_n_per_genre(rs, "Rock", n=100)
print("Rock head(5):")
display(top100_rock[["artist", "title", "play_count"]].head(5))
print("\nRock tail(5):")
display(top100_rock[["artist", "title", "play_count"]].tail(5))

Rock head(5):


,artist,title,play_count
0,Björk,Undo,648239.0
1,Kings Of Leon,Revelry,527893.0
2,Harmonia,Sehr kosmisch,425463.0
3,OneRepublic,Secrets,292642.0
4,Tub Ring,Invalid,268353.0



Rock tail(5):


,artist,title,play_count
95,Metric,Gold Guns Girls,28148.0
96,Pearl Jam,Encore Break,27579.0
97,Daughtry,No Surprise,27187.0
98,Eric Clapton,Tears In Heaven,26999.0
99,Nick Lowe,All Men Are Liars,26683.0


In [6]:
top100_rap = top_n_per_genre(rs, "Rap", n=100)
print("Rap head(5):")
display(top100_rap[["artist", "title", "play_count"]].head(5))
print("\nRap tail(5):")
display(top100_rap[["artist", "title", "play_count"]].tail(5))

Rap head(5):


,artist,title,play_count
0,Alliance Ethnik,Représente,241669.0
1,Beastie Boys,The Maestro,72381.0
2,Eminem,Without Me,63918.0
3,Black Eyed Peas,Imma Be,62438.0
4,Kid Cudi,Up Up & Away,59810.0



Rap tail(5):


,artist,title,play_count
95,Shwayze,Buzzin',7384.0
96,Orishas,El Kilo,7324.0
97,Snoop Dogg,Sexual Eruption,7171.0
98,Bone Thugs-N-Harmony,Tha Crossroads,7124.0
99,Orishas,Habana,6998.0


In [7]:
top100_elec = top_n_per_genre(rs, "Electronic", n=100)
print("Electronic head(5):")
display(top100_elec[["artist", "title", "play_count"]].head(5))
print("\nElectronic tail(5):")
display(top100_elec[["artist", "title", "play_count"]].tail(5))

Electronic head(5):


,artist,title,play_count
0,Southside Spinners,Luvstruck,84225.0
1,The Black Keys,Tighten Up,81179.0
2,Deadmau5,Ghosts 'n' Stuff (Original Instrumental Mix),63951.0
3,Daft Punk,Harder Better Faster Stronger,63170.0
4,Clara Hill,Clara meets Slope - Hard To Say,58887.0



Electronic tail(5):


,artist,title,play_count
95,Nicolette,No Government,9541.0
96,Two Door Cinema Club,Eat That Up_ It's Good For You,9524.0
97,Moby,Why Does My Heart Feel So Bad? (2006 Digital R...,9491.0
98,Death In Vegas,Girls,9490.0
99,Johan Gielen,Flash,9431.0


In [8]:
# a single user's favourite Rock songs
user_1 = rs.triplets["user_id"].unique()[0]
print(f"User 1 : {user_1}")

top100_rock_user_1 = top_n_per_genre(rs, "Rock", n=100, user_id=user_1)
display(top100_rock_user_1[["artist", "title", "play_count"]].head(10))

User 1 : b80344d063b5ccb3212f76538f3d9e43d87dca9e


,artist,title,play_count
0,Héroes del Silencio,Apuesta Por El Rock 'N' Roll,5.0
1,Chris Bell,Speed Of Sound,2.0
2,John Mayer,Love Song For No One,1.0
3,Josh Rouse,Nice To Fit In,1.0
4,Josh Rouse,Domesticated Lovers,1.0
5,Andrew Bird,Oh No,1.0
6,Takka Takka,Fever,1.0
7,Incubus,Drive,1.0
8,Josh Rouse,It's The Night Time,1.0
9,Jimmy Eat World,The Middle,1.0


---
## 3. Collections — 50 songs about a keyword

Lyrics are loaded as a tidy `(track_id, word, count)` table (~210 k tracks ×
5000 vocabulary words). The musiXmatch vocabulary is Porter-stemmed, so
`happiness` → `happi`, `loneliness` → `loneli`. Three approaches are compared:
baseline keyword counts, word2vec-expanded keywords, and ML classification.

In [9]:
KEYWORDS = ["love", "war", "happiness", "loneliness", "money"]

In [10]:
baseline_results = collection_baseline(rs, KEYWORDS, n=1, top_n=50)
for kw in KEYWORDS:
    print(f"\nKeyword: {kw}")
    display(baseline_results.get(kw, "No results found"))
print(f"peak RSS : {rss_mb()} MB")


Keyword: love


,artist,title,play_count
1,OneRepublic,Secrets,292642.0
2,Five Iron Frenzy,Canada,274627.0
3,Tub Ring,Invalid,268353.0
4,Train,Hey_ Soul Sister,209212.0
5,Angels and Airwaves,The Gift,192884.0
6,Train,Marry Me,174080.0
7,Lil Wayne / Eminem,Drop The World,155717.0
8,Bill Withers,Make Love To Your Mind,146978.0
9,Pavement,Mercy:The Laundromat,130116.0
10,Travie McCoy,Billionaire [feat. Bruno Mars] (Explicit Albu...,122318.0



Keyword: war


,artist,title,play_count
1,Taylor Swift,Love Story,89589.0
2,Miley Cyrus,Party In The U.S.A.,78443.0
3,Taylor Swift,You Belong With Me,65582.0
4,Five Finger Death Punch,Bad Company,42348.0
5,Black Eyed Peas,Pump It,41243.0
6,Jack Johnson,Breakdown,32684.0
7,Eddy Grant,Electric Avenue,29781.0
8,Alicia Keys,Empire State Of Mind (Part II) Broken Down,25203.0
9,Beirut,Elephant Gun,25133.0
10,John Mayer,Perfectly Lonely,24737.0



Keyword: happiness


...


Keyword: loneliness


,artist,title,play_count
1,O.G.C.,Gunn Clapp,33897.0
2,Toby Keith,American Soldier,23872.0
3,Rise Against,Hero Of War,18939.0
4,Vampire Weekend,Horchata,16823.0
5,The Kooks,She Moves In Her Own Way,14561.0
6,Dierks Bentley,Sideways,13892.0
7,Eric Church,Love Your Love The Most,10738.0
8,Iron And Wine,The Devil Never Sleeps (Album),10596.0
9,Eric Church,Guys Like Me,8627.0
10,311,Get Down,8258.0



Keyword: money


,artist,title,play_count
1,Paris Combo,Prête A Porter,17179.0
2,Yann Tiersen,A Quai,12471.0
3,Calle 13 Featuring Café Tacuba,No Hay Nadie Como Tú,9591.0
4,Sondre Lerche,Hoisting The Flag,9024.0
5,Mylène Farmer,Porno Graphique,7840.0
6,La Rue Ketanou,Les derniers aventuriers,7575.0
7,Clarika,Les Garçons Dans Les Vestiaires,7407.0
8,Kate Ryan,Désenchantée,6071.0
9,Macaco,Mama Tierra,5591.0
10,Metric,Poster Of A Girl,5300.0


peak RSS : 5202 MB


In [11]:
w2v_results = collection_word2vec(rs, KEYWORDS, n=50, top_n=50)
for kw, df in w2v_results.items():
    print(f"\n{'='*50}\n  Collection: {kw.upper()}\n{'='*50}")
    display(df)


  Collection: LOVE


,artist,title,play_count
1,Travie McCoy,Billionaire [feat. Bruno Mars] (Explicit Albu...,122318.0
2,Beastie Boys,Unite (2009 Digital Remaster),99137.0
3,Miley Cyrus,Party In The U.S.A.,78443.0
4,Eminem,Mockingbird,74103.0
5,Eminem,Without Me,63918.0
6,Guns N' Roses,Paradise City,60787.0
7,Reality Check,Masquerade (Reality Check Album Version),48393.0
8,Eminem / Nate Dogg,'Till I Collapse,44305.0
9,Guns N' Roses,Don't Cry (Original),40480.0
10,Gang Starr/Inspectah Deck,Above The Clouds (Edited),39140.0



  Collection: WAR


,artist,title,play_count



  Collection: HAPPINESS


,artist,title,play_count
1,Spice Girls,Something Kinda Funny,801.0
2,New Radicals,Maybe You've Been Brainwashed Too,495.0
3,The Roots,The Session (Longest Posse Cut In History_ 12:43),298.0
4,Obie Trice,Never Forget Ya,101.0
5,Lil' Romeo,The Girlies,13.0
6,Heather Small,I've Been There,12.0
7,Jessica Simpson,My Wonderful,8.0
8,Missing Persons,Give (Dance Mix) (2002 Digital Remaster),3.0
9,Jessica Simpson,I've Got My Eyes On You,0.0
10,Red Hot Chili Peppers,Around The World (Album Version),0.0



  Collection: LONELINESS


,artist,title,play_count
1,Silkk The Shocker,D-Game (feat. Master P_ Krazy and Terror) (Remix),0.0



  Collection: MONEY


,artist,title,play_count
1,The Hollies,I'm Down,3973.0
2,Bruce Springsteen,I'm Goin' Down,2371.0
3,Andres Calamaro,No Se Puede Vivir Del Amor,1322.0
4,Will.I.Am,Over,1260.0
5,Red Hot Chili Peppers,I Like Dirt (Album Version),686.0
6,Spice Girls,Saturday Night Divas,478.0
7,Jamie Foxx featuring Gucci Mane,Speak French,349.0
8,Simply Red,Model,288.0
9,Fatboy Slim,The World Went Down,49.0
10,G-Unit,Get Down,36.0


In [12]:
classifiers = ["nb", "logistic", "sgd", "forest"]
clf_results = collection_classification_compare(rs, KEYWORDS, n=10, neg_ratio=1, classifiers=classifiers)
for name in classifiers:
    print(f"\n{'='*60}\nClassifier: {name.upper()}\n{'='*60}")
    for kw, df in clf_results[name].items():
        print(f"\n  Collection: {kw.upper()}")
        display(df)
print(f"peak RSS : {rss_mb()} MB")


Classifier: NB

  Collection: LOVE


,artist,title,play_count
1,Blessid Union Of Souls,Could've Been With You,57.0
2,Maxwell,Arroz Con Pollo,0.0
3,Maxwell,Eachhoureachsecondeachminuteeachday:Of My Life,0.0
4,Joe Clay,Annabella,0.0
5,Maxwell,This Woman's Work,3601.0
6,Whitesnake,Mistreated (Live) (2007 Digital Remaster),383.0
7,Ernest Tubb,Have You Ever Been Lonely (Have You Ever Been ...,4.0
8,Glenn Hughes,Mistreated,0.0
9,Hughes Turner Project,Mistreated,0.0
10,Songs:Ohia,Ghost Tropic,94.0



  Collection: WAR


,artist,title,play_count
1,Kenny G,Santa Claus Is Coming To Town,8.0
2,All-4-One,Santa Claus Is Coming To Town (LP Version),0.0
3,Matt Belsante,Santa Claus Is Coming To Town (White Christmas...,0.0
4,Willie Nelson,Here Comes Santa Claus,0.0
5,Gene Autry,Here Comes Santa Claus,0.0
6,Bob Dylan,Here Comes Santa Claus,0.0
7,Andrea Bocelli,Santa Claus Is Coming To Town,36.0
8,The SA,Santa Claus Is Coming to Town,71.0
9,Patty Loveless,The Boys Are Back In Town,0.0
10,Everclear,The Boys Are Back In Town,836.0



  Collection: HAPPINESS


...


  Collection: LONELINESS


,artist,title,play_count
1,Company Flow,8 Steps To Perfection,802.0
2,Mobb Deep,Backstage Pass,834.0
3,Kool Moe Dee,I'm Hittin' Hard,0.0
4,Patti Smith Group,Piss Factory,254.0
5,Extreme,Wind Me Up,94.0
6,Sutton Foster,Flight,76.0
7,Nelly,Batter Up (Full Phat Remix),0.0
8,Kool G Rap,Ghetto Knows,0.0
9,Krayzie Bone,I Don't Give A F**k,0.0
10,Albert King,Blues Power,0.0



  Collection: MONEY


,artist,title,play_count
1,La Fouine,Ma Tabatière (Chronique D'Un Dealer),182.0
2,Anis,La Preuve Par 1000 (Mahlich Boy),96.0
3,Sniper,Y'a Pas De Mérite,78.0
4,Seth Gueko,J'oublierai Pas,52.0
5,Abd Al Malik,Soldat De Plomb,48.0
6,Sniper,Hall Story,45.0
7,Le Klub des 7,L'appel,45.0
8,Lara Fabian,Comme Ils Disent,40.0
9,Abd Al Malik,La Gravité,37.0
10,Sniper,Visions Chaotiques,27.0


peak RSS : 5412 MB


In [13]:
# overlap summary between the three approaches
def summarize(df):
    if df is None or len(df) == 0:
        return 0, set()
    return len(df), set(zip(df["artist"], df["title"]))

methods = [("baseline", baseline_results), ("w2v", w2v_results)]
methods += [(f"clf_{name}", clf_results[name]) for name in classifiers]

rows = {}
for kw in KEYWORDS:
    rows[kw.upper()] = {label: summarize(res.get(kw))[0] for label, res in methods}
display(__import__("pandas").DataFrame(rows).T)

,baseline,w2v,clf_nb,clf_logistic,clf_sgd,clf_forest
LOVE,50,50,50,50,50,50
WAR,50,0,50,50,50,50
HAPPINESS,50,14,50,50,50,50
LONELINESS,50,1,50,50,50,50
MONEY,50,33,50,50,50,50


---
## 4. Collaborative filtering (implicit ALS)

Matrix factorization (Hu, Koren & Volinsky 2008) on the sparse
user×song matrix, evaluated with precision@10.


In [14]:
# free the lyrics table before ALS so peak memory stays low
del rs.lyrics_long
gc.collect()

user_item, user_idx, song_idx, idx_song = build_user_item_matrix(rs)
print(f"user×song matrix : {user_item.shape}  (nnz={user_item.nnz:,})")
print(f"peak RSS         : {rss_mb()} MB")

user×song matrix : (1019318, 384546)  (nnz=48,373,586)
peak RSS         : 5716 MB


In [15]:
train, test = implicit_split(user_item, train_percentage=0.8, random_state=42)
print(f"train nnz : {train.nnz:,}")
print(f"test  nnz : {test.nnz:,}")

train nnz : 38,702,352
test  nnz : 9,671,234


In [ ]:
model = fit_als(train, factors=192, regularization=0.09, alpha=1.0, iterations=25)
print(f"peak RSS : {rss_mb()} MB")

/home/samy/MySpotify/.venv/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 28 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
 16%|█▌        | 4/25 [00:32<02:43,  7.78s/it]

### 4a — User-based recommendations (latent factors)

The eval user `b7815dbb206eb2831ce0fe040d0aa537e2e800f7` — the list must
contain 10 tracks the user has **not** already listened to.


In [ ]:
EVAL_USER = "b7815dbb206eb2831ce0fe040d0aa537e2e800f7"
recs_user = recommend_users_df(
    EVAL_USER, model, user_item, user_idx, idx_song, rs.tracks, top_n=10
)
print(f"10 recommendations for user {EVAL_USER}:")
display(recs_user)

heard = set(rs.triplets.loc[rs.triplets["user_id"] == EVAL_USER, "song_id"])
uid = user_idx[EVAL_USER]
new_ids, _ = model.recommend(uid, user_item[uid], N=10)
new = {idx_song[i] for i in new_ids}
print(f"already-listened tracks in the list: {len(heard & new)} (must be 0)")

### 4b — Similar tracks (item-based CF)

The eval track `SOWYSKH12AF72A303A` — the list must contain 10 tracks and must
**not** contain the seed itself.


In [ ]:
EVAL_SONG = "SOWYSKH12AF72A303A"
recs_item = recommend_tracks_df(
    EVAL_SONG, model, song_idx, idx_song, rs.tracks, top_n=10
)
print(f"10 similar tracks to {EVAL_SONG}:")
display(recs_item)
sid = song_idx[EVAL_SONG]
sim_ids, _ = model.similar_items(sid, N=11)
seed_in_list = any(idx_song[i] == EVAL_SONG for i in sim_ids[1:])
print(f"seed in the list: {seed_in_list} (must be False)")

### Precision@10 — user-based CF

`precision@10` must be greater than 10%.


In [ ]:
pk = evaluate_user_cf(model, train, test, top_n=10)
print(f"mean precision@10 : {pk:.4f} (> 0.10)  -> {'PASS' if pk > 0.10 else 'FAIL'}")
print(f"peak RSS          : {rss_mb()} MB")

---
## Done

All four tasks run on pandas DataFrames parsed **in place from the uncleaned raw
files** (`data/raw`) — nothing is cleaned or saved to disk.

| Task | Output | RSS |
|------|--------|-----|
| Top-250 | 250 most-played songs (global + per-user) | see prints above |
| Top-100 by genre | Rock / Rap / Electronic lists | |
| Collections | 50 songs per keyword (baseline, w2v, classification) | |
| Collaborative filtering | ALS model, user & item recommendations, p@k | |